# Trabajo Práctico Final — Introducción a la Ingeniería de Datos

**Autora:** Bioing. Aylin Agatha Vazquez Chenlo — Doctorado en Ingeniería, ITBA

**Profesor:** Dr. Nicolás Guillermo Tripp

**Repositorio:** https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos

Cada estudiante deberá entregar, con al menos 1 semana de anticipación a la defensa, un enlace a un cuaderno de Google Colab público dentro del repositorio en GitHub declarado en el cursado, debidamente documentado mediante celdas de texto Markdown y estrictamente reproducible.

El cuaderno debe cumplir obligatoriamente con los siguientes requisitos:

## **Declaración de Uso de Inteligencia Artificial (IA)**

El uso de IA generativa está permitido y fomentado, siempre que se declare de forma transparente. El cuaderno debe incluir una celda de texto al inicio titulada "Declaración de uso de IA", listando cada herramienta utilizada y su propósito específico.

- Formato exigido: "[Nombre y versión de la IA]" para "[Propósito exacto]".
- Ejemplos: "Claude Opus 4.6" para "mejorar redacción de textos de las justificaciones", "Chat GPT 5.6" para "crear código fuente para expresiones regulares", o "GitHub Copilot" para "autocompletar código de gráficos en Plotly".

**Instrucciones de la IA**: Debe presentar una guía de instrucciones que permita reproducir lo mas fiel posible la información generada mediante inteligencia artificial. Esta guía debe demostrar la capacidad del estudiante para dirigir a la herramienta en al creación del resultado obtenido (Human-in-the-loop). Se espera que las instrucciones incluyan el vocabulario especifico del curso.
No se aceptan instrucciones que sean una copia directa de los enunciados de actividades.

## **Datos**

Debe utilizar un dataset crudo disponible desde un repositorio en la nube o fuente pública (GitHub, Zenodo, OSF, API, etc.). El dataset debe ser de un tema afín a su trabajo de tesis. No se aceptarán cuadernos que requieran subir archivos manualmente o que dependan de rutas locales/privadas (ej. carpetas privadas de Google Drive).

## **Extracción, Transformación y Carga (ETL)**

Se debe realizar el proceso completo de ETL, incluyendo:

- Carga de datos optimizando los parámetros (elección de columnas, tipado de variables).
- Identificación y resolución de valores faltantes (NAs/NULLs) y datos atípicos (outliers). Cada decisión de imputación, transformación o descarte debe estar justificada metodológicamente en celdas de texto.
- Almacenamiento de los datos en una base datos transaccional SQLite normalizada como mínimo en 2NF.
- Creación y carga de los datos en un almacén de datos SQLite, siguiendo los principios del modelado dimensional (esquema estrella o copo de nieve), separando claramente las tablas de hechos de las tablas de dimensiones.

## **Visualización**

Utilizando los datos del almacén deberá construir una interfaz interactiva de visualización (streamlit, posix, shiny, dash, postly) que transmita un mensaje claro sobre su investigación.

## **Límite de páginas** 
15

---

**Utilización de IA para co-desarrollo:** Se utilizó **Claude Opus 5.5** (Anthropic) para:
- Corregir la redacción de las explicaciones/justificaciones en las celdas markdown del notebook.

---

## Contexto de investigación

**Dataset elegido:** *The Bitbrain Open Access Sleep (BOAS) dataset* (López-Larraz et al.), OpenNeuro `ds005555` v1.1.3, DOI [10.18112/openneuro.ds005555.v1.1.3](https://doi.org/10.18112/openneuro.ds005555.v1.1.3), licencia CC0.
- 128 noches de 100 personas adultas sanas, registradas en simultáneo con una PSG clínica (Micromed) y una vincha de EEG wearable (Bitbrain).
- Cada época de 30 s tiene tres etiquetas de estadio: el consenso de tres expertos sobre la PSG (criterios AASM), la estadificación automática con IA sobre la PSG y la estadificación automática con IA sobre la vincha.
- Los datos están en formato [BIDS](https://bids-specification.readthedocs.io/en/stable/): `participants.tsv` (edad, sexo, IMC) y un `events.tsv` por noche y dispositivo.

**Pregunta:** ¿Qué tan bien estadifica el sueño un dispositivo wearable de EEG frente al estándar clínico (polisomnografía puntuada por expertos), y cómo varía esa concordancia según el estadio de sueño y las características de la persona (edad, sexo, IMC)?

**Relación con mi tesis:** Mi tesis aborda el procesamiento de señales de sueño, principalmente EEG. Validar dispositivos portátiles contra la polisomnografía (PSG) es un paso necesario para llevar el monitoreo del sueño fuera del laboratorio.

---

## Set Up

Instalación de las dependencias que no vienen preinstaladas en Google Colab y carga de librerías. La lista completa de dependencias está en [`TP-Final/requirements.txt`](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/requirements.txt): si el archivo está disponible localmente (ejecución desde el repositorio clonado) se usa esa copia; si no (Colab), se descarga desde GitHub.

In [1]:
# Instalación de dependencias
import os, urllib.request

REQ_URL = "https://raw.githubusercontent.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/main/TP-Final/requirements.txt"
REQ_PATH = "requirements.txt"

if not os.path.exists(REQ_PATH):
    urllib.request.urlretrieve(REQ_URL, REQ_PATH)  # En Colab el repositorio no está clonado

%pip install -q -r {REQ_PATH}


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Importación de librerías
# Librería estándar
import os
import sqlite3
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# Ingesta y manipulación
import numpy as np
import pandas as pd
import requests

# Análisis
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

# Visualización interactiva
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, dcc, html, Input, Output

# Configuración general
SEED = 42                                   # Semilla para reproducibilidad
np.random.seed(SEED)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 2)   # float32 se muestra sin ruido de redondeo
EN_COLAB = "google.colab" in str(get_ipython())

import sklearn, plotly, dash
print(f"Entorno: {'Google Colab' if EN_COLAB else 'local'}")
for lib in (pd, np, sklearn, plotly, dash):
    print(f"  {lib.__name__:<8} {lib.__version__}")

Entorno: local
  pandas   3.0.5
  numpy    2.5.3
  sklearn  1.9.1
  plotly   7.1.0
  dash     4.5.0


---

## 1. Extracción

**Fuente.** Los archivos se descargan desde la réplica oficial de OpenNeuro en GitHub ([`OpenNeuroDatasets/ds005555`](https://github.com/OpenNeuroDatasets/ds005555)), fijando el *tag* `1.1.3` en lugar de la rama `main`: cualquier ejecución futura lee exactamente la misma versión publicada, aunque el dataset se actualice.

**Alcance.** Se descargan `participants.tsv` (una fila por noche registrada) y los 256 `events.tsv` (128 noches × 2 dispositivos), que contienen la estadificación por época de 30 s. Las señales crudas (`.edf`, ~180 MB por noche) **no** se descargan porque la pregunta de investigación se responde con las etiquetas de estadio.

**Caché.** Cada archivo se guarda en `data/raw/ds005555/`; si ya existe, no se vuelve a descargar.

In [3]:
# Parámetros de la fuente y estructura de carpetas
DATASET = "ds005555"
VERSION = "1.1.3"   # tag de la versión publicada en OpenNeuro
BASE_URL = f"https://raw.githubusercontent.com/OpenNeuroDatasets/{DATASET}/{VERSION}"

DIR_RAW = Path("data/raw") / DATASET
DIR_PROCESSED = Path("data/processed")
for carpeta in (DIR_RAW, DIR_PROCESSED):
    carpeta.mkdir(parents=True, exist_ok=True)

def descargar(ruta_relativa: str) -> Path:
    """Descarga un archivo del dataset si no está en caché y devuelve su ruta local."""
    destino = DIR_RAW / ruta_relativa
    if not destino.exists():
        destino.parent.mkdir(parents=True, exist_ok=True)
        respuesta = requests.get(f"{BASE_URL}/{ruta_relativa}", timeout=30)
        respuesta.raise_for_status()  # Falla explícitamente ante un error HTTP
        destino.write_bytes(respuesta.content)
    return destino

def memoria_mb(df: pd.DataFrame) -> float:
    """Memoria RAM ocupada por un DataFrame (incluye el contenido de los strings)."""
    return df.memory_usage(deep=True).sum() / 1024**2

### 1.1 Participantes (`participants.tsv`)

Por convención BIDS, `participant_id` identifica a cada **noche registrada** (`sub-1` … `sub-128`), mientras que `pid` identifica a la **persona** (algunas personas durmieron más de una noche). Los parámetros de carga se eligen a partir del diccionario de datos (`participants.json`):

| Columna | Tipo | Justificación |
|---|---|---|
| `participant_id` | `string` | Identificador textual de la sesión. |
| `pid` | `Int16` | Entero nullable; 100 personas. |
| `age` | `Int8` | Edad en años (18–82) cabe en 8 bits; *nullable* para admitir faltantes. |
| `sex` | `category` | Solo dos niveles (`F`, `M`). |
| `bmi` | `float32` | Body-mass-index de la persona. |

`na_values=["n/a"]` interpreta explícitamente el marcador de faltante que usa BIDS.

In [4]:
descargar("participants.json")  # Diccionario de variables (metadatos)

# Descarga y carga de los datos de participantes
participantes = pd.read_csv(
    descargar("participants.tsv"),
    sep="\t",
    usecols=["participant_id", "pid", "age", "sex", "bmi"],
    dtype={"participant_id": "string", "pid": "Int16", "age": "Int8",
           "sex": "category", "bmi": "float32"},
    na_values=["n/a"],
)
print(f"{len(participantes)} noches registradas | {participantes['pid'].nunique()} personas | "
      f"{memoria_mb(participantes):.3f} MB")
participantes.head()

128 noches registradas | 100 personas | 0.003 MB


,participant_id,age,sex,bmi,pid
0,sub-1,27,F,19.8,89
1,sub-2,27,F,19.8,89
2,sub-3,27,M,25.1,1
3,sub-4,26,F,22.0,2
4,sub-5,27,F,20.7,3


### 1.2 Hipnogramas (`events.tsv`)

Cada noche tiene un archivo por dispositivo: `acq-psg` (polisomnografía) y `acq-headband` (vincha wearable). Ambos tienen una fila por época de 30 s con las columnas `onset`, `duration`, `begsample`, `endsample`, `offset` y las etiquetas de estadio (`stage_hum`, solo en PSG; `stage_ai`). Las descargas se hacen en paralelo porque son operaciones de entrada/salida.

In [5]:
SESIONES = participantes["participant_id"].tolist()
ADQUISICIONES = ["psg", "headband"]

def ruta_eventos(sesion: str, acq: str) -> str:
    return f"{sesion}/eeg/{sesion}_task-Sleep_acq-{acq}_events.tsv"

with ThreadPoolExecutor(max_workers=8) as pool:
    archivos = list(pool.map(descargar, [ruta_eventos(s, a) for s in SESIONES for a in ADQUISICIONES]))
print(f"{len(archivos)} archivos de eventos disponibles en {DIR_RAW}/")

256 archivos de eventos disponibles en data/raw/ds005555/


Primero se cargan los archivos con la configuración por defecto de pandas para verificar que las columnas que se van a descartar son redundantes:

- `duration` vale siempre 30 s y `offset` siempre 0 (constantes, sin información).
- `begsample` y `endsample` se deducen de `onset` y de la frecuencia de muestreo (256 Hz, declarada en los `*_eeg.json`).

Luego se cargan solo `onset` y las etiquetas, con enteros de 8 y 32 bits.

La reducción de memoria entre ambas cargas combina tres efectos:
1. **Proyección de columnas:** de 7 columnas a 3 (PSG) o 2 (vincha).
2. **Tipado reducido:** `int32` para `onset` y `Int8` (entero que admite nulos) para los estadios, en lugar de `int64`.
3. **Unión de dispositivos:** al unir PSG y vincha por época, `onset` se almacena una sola vez y las filas se reducen a la mitad.

Los dos primeros corresponden a la optimización de parámetros de carga; el tercero es una consecuencia de reorganizar la tabla.

In [7]:
FS = 256  # Hz, frecuencia de muestreo declarada en cada *_eeg.json

In [10]:
def leer_eventos(sesion: str, acq: str, optimizado: bool = True) -> pd.DataFrame:
    ruta = DIR_RAW / ruta_eventos(sesion, acq)
    if not optimizado:
        return pd.read_csv(ruta, sep="\t")
    columnas = ["onset", "stage_hum", "stage_ai"] if acq == "psg" else ["onset", "stage_ai"]
    return pd.read_csv(ruta, sep="\t", usecols=columnas, na_values=["n/a"],
                       dtype={"onset": "int32", "stage_hum": "Int8", "stage_ai": "Int8"})

# Carga por separado: todas las columnas con tipos por defecto (int64)
por_separado = pd.concat({(s, a): leer_eventos(s, a, optimizado=False)
                     for s in SESIONES for a in ADQUISICIONES},
                    names=["participant_id", "acq", "fila"])

# Validación de las columnas que se descartan
assert (por_separado["duration"] == 30).all()
assert (por_separado["offset"] == 0).all()
assert (por_separado["begsample"] == por_separado["onset"] * FS + 1).all()
assert (por_separado["endsample"] == (por_separado["onset"] + 30) * FS).all()
print("✓ duration, offset, begsample y endsample son redundantes con onset")

✓ duration, offset, begsample y endsample son redundantes con onset


In [11]:
# Carga optimizada: proyección de columnas + tipos reducidos
def cargar_dispositivo(acq: str) -> pd.DataFrame:
    df = pd.concat({s: leer_eventos(s, acq) for s in SESIONES}, names=["participant_id", "fila"])
    return df.reset_index(level="participant_id").reset_index(drop=True)

psg = cargar_dispositivo("psg").rename(columns={"stage_ai": "stage_ai_psg"})
vincha = cargar_dispositivo("headband").rename(columns={"stage_ai": "stage_ai_hb"})

# Unión época a época: cada época debe existir en ambos dispositivos (relación 1:1)
eventos = psg.merge(vincha, on=["participant_id", "onset"], how="outer",
                    validate="one_to_one", indicator=True)
assert (eventos["_merge"] == "both").all(), "Hay épocas presentes en un solo dispositivo"
eventos = eventos.drop(columns="_merge")
eventos["participant_id"] = eventos["participant_id"].astype("category")

print(f"Carga por separado   : {memoria_mb(por_separado):6.2f} MB ({len(por_separado):,} filas, PSG y vincha por separado)")
print(f"Carga optimizada: {memoria_mb(eventos):6.2f} MB ({len(eventos):,} épocas, PSG y vincha unidos)")
del por_separado

Carga por separado   :  13.98 MB (240,190 filas, PSG y vincha por separado)
Carga optimizada:   1.38 MB (120,095 épocas, PSG y vincha unidos)


### 1.3 Tabla única de extracción y controles de integridad

Se unen las épocas con los atributos de cada noche en una única tabla plana (*One Big Table*), que es el punto de partida de la Transformación. Antes de avanzar se verifican las reglas de integridad que el resto del pipeline asume.

In [12]:
obt = eventos.merge(participantes.astype({"participant_id": "category"}),
                    on="participant_id", how="left", validate="many_to_one")

# Controles de integridad
assert obt["participant_id"].nunique() == 128 and obt["pid"].nunique() == 100
assert obt.groupby("participant_id", observed=True)["onset"].diff().dropna().eq(30).all(), \
    "Hay saltos entre épocas consecutivas"
assert set(obt["stage_hum"].unique()) <= {0, 1, 2, 3, 4, 8}
assert set(obt["stage_ai_psg"].unique()) | set(obt["stage_ai_hb"].unique()) <= {-2, 0, 1, 2, 3, 4}
print("✓ 128 noches, 100 personas, épocas contiguas y códigos de estadio dentro del dominio")

obt.to_parquet(DIR_PROCESSED / "01_extraccion_obt.parquet", index=False)
print(f"OBT: {obt.shape[0]:,} filas × {obt.shape[1]} columnas | {memoria_mb(obt):.2f} MB")
obt.head()

✓ 128 noches, 100 personas, épocas contiguas y códigos de estadio dentro del dominio
OBT: 120,095 filas × 9 columnas | 2.52 MB


,participant_id,onset,stage_hum,stage_ai_psg,stage_ai_hb,age,sex,bmi,pid
0,sub-1,0,3,3,3,27,F,19.8,89
1,sub-1,30,3,3,3,27,F,19.8,89
2,sub-1,60,3,3,3,27,F,19.8,89
3,sub-1,90,3,3,3,27,F,19.8,89
4,sub-1,120,3,3,3,27,F,19.8,89


### 1.4 Primer diagnóstico de calidad

En esta etapa solo se identifican los problemas; su resolución se justifica en la sección de Transformación.

In [13]:
CODIGOS = {"stage_hum": 8, "stage_ai_psg": -2, "stage_ai_hb": -2}  # 8 = desconexión del PSG, -2 = artefacto
print("Faltantes en participants.tsv:")
print(participantes[participantes.isna().any(axis=1)].to_string(index=False))

print("\nÉpocas con código especial (no son un estadio de sueño):")
for col, cod in CODIGOS.items():
    n = (obt[col] == cod).sum()
    print(f"  {col:<13} = {cod:>2}: {n:>5,} épocas ({n / len(obt):.1%})")

horas = obt.groupby("participant_id", observed=True).size() * 30 / 3600
print(f"\nDuración del registro por noche (h): mediana {horas.median():.1f} | "
      f"mín {horas.min():.1f} ({horas.idxmin()}) | máx {horas.max():.1f}")

primer_estadio = obt.groupby("participant_id", observed=True)["stage_hum"].first()
print(f"Noches que no comienzan en vigilia: {primer_estadio[primer_estadio != 0].to_dict()}")

por_persona = participantes.groupby("pid")[["age", "bmi"]].nunique()
print("\nPersonas con más de un valor de edad o IMC entre noches:")
print(participantes[participantes["pid"].isin(por_persona[(por_persona > 1).any(axis=1)].index)]
      .to_string(index=False))

Faltantes en participants.tsv:
participant_id  age sex  bmi  pid
        sub-16 <NA>   F 26.8   11
        sub-29   23   F  NaN   21

Épocas con código especial (no son un estadio de sueño):
  stage_hum     =  8:   336 épocas (0.3%)
  stage_ai_psg  = -2:   550 épocas (0.5%)
  stage_ai_hb   = -2: 4,881 épocas (4.1%)

Duración del registro por noche (h): mediana 7.8 | mín 3.4 (sub-8) | máx 9.3
Noches que no comienzan en vigilia: {'sub-1': 3}

Personas con más de un valor de edad o IMC entre noches:
participant_id  age sex  bmi  pid
        sub-23   34   M 28.1   15
       sub-124   35   M 27.3   15


**Hallazgos** (que se resuelven en la Transformación):

1. **Faltantes:** Una noche sin edad (`sub-16`) y una sin IMC (`sub-29`).
2. **Códigos especiales:** Las etiquetas `8` (desconexión del PSG, p. ej. para ir al baño) y `-2` (artefacto o dato faltante en la IA) no son estadios de sueño. Son NAs codificados como números y se tratan como faltantes. La vincha concentra la mayor proporción de artefactos.
3. **Atípico de duración:** `sub-8` registra 3,4 h, muy por debajo de la mediana (~7,8 h).
4. **Inicio de registro:** `sub-1` comienza directamente en N3 (las otras 127 noches comienzan en vigilia), lo que indica que el registro empezó con la persona ya dormida; su latencia de sueño no es calculable.
5. **Dependencias:** El sexo es constante por persona, pero la edad y el IMC pueden cambiar entre noches de la misma persona (`pid` 15). Esto guía la normalización: el sexo es atributo de la persona; la edad y el IMC, de la noche.

---

## 2. Transformación

***Objetivo:*** Resolver los problemas detectados en el diagnóstico y derivar las variables que necesitan los modelos transaccional y dimensional. El criterio general tomado es que no se elimina información cruda. Los valores inválidos se convierten en faltantes explícitos y las noches problemáticas se marcan con reglas documentadas, para que cada análisis decida qué incluir.

### 2.1 Códigos especiales → Faltantes y variables temporales

Los códigos `8` (desconexión del PSG) y `-2` (artefacto) son faltantes codificados como números: si se dejaran, cualquier promedio o conteo por estadio los trataría como un estadio más. Se reemplazan por `NA`. No se imputan: asignar un estadio a una época sin señal equivale a inventar una etiqueta clínica, y la propia tasa de faltantes es un indicador de calidad del dispositivo que conviene conservar.

Se agregan además el número de época, la hora desde el inicio del registro y el tercio de la noche (contexto temporal para el almacén).

In [20]:
COLS_ESTADIO = ["stage_hum", "stage_ai_psg", "stage_ai_hb"]
epocas = obt[["participant_id", "onset"] + COLS_ESTADIO].copy()
for col, codigo in CODIGOS.items():          # CODIGOS se definió en el diagnóstico (1.4)
    epocas[col] = epocas[col].mask(epocas[col] == codigo)

epocas["epoca"] = (epocas["onset"] // 30).astype("int16")
epocas["hora"] = (epocas["onset"] // 3600).astype("int8")
n_epocas = epocas.groupby("participant_id", observed=True)["epoca"].transform("size")
epocas["tercio"] = (epocas["epoca"] * 3 // n_epocas + 1).astype("int8")

print("Faltantes explícitos por columna de estadio:")
print(epocas[COLS_ESTADIO].isna().sum().to_string())

Faltantes explícitos por columna de estadio:
stage_hum        336
stage_ai_psg     550
stage_ai_hb     4881


### 2.2 Faltantes en edad e IMC

Antes de considerar una imputación estadística se busca una imputación deductiva ya que si la misma persona (`pid`) tuviera otra noche con el dato informado, se podría completar sin error. Pero este es el caso porque ambas personas `sub-16` y `sub-29` registraron una sola noche.

**Decisión: *no imputar*.** Son 2 valores en 256 (0.8% por variable) y el mecanismo es compatible con una omisión al azar en un dato autorreportado. Edad e IMC se usan para agrupar noches, no como medidas: imputar por mediana asignaría a la persona a un grupo de forma arbitraria. Decido conservarlo como `NULL` en la base transaccional y como categoría "Sin dato" en el almacén (lo cual es una práctica estándar del modelado dimensional para miembros desconocidos).

Se derivan el grupo etario y la categoría de IMC según los cortes de la OMS.

In [21]:
noches = participantes.copy()
faltantes = noches.loc[noches[["age", "bmi"]].isna().any(axis=1), "pid"]
otras_noches = noches[noches["pid"].isin(faltantes)].groupby("pid").size()
print(f"Noches registradas por las personas con faltantes: { {int(k): int(v) for k, v in otras_noches.items()} }")  # 1 → no hay deducción posible

noches["grupo_etario"] = pd.cut(noches["age"], bins=[17, 30, 45, 60, 120],
                                labels=["18-30", "31-45", "46-60", ">60"])
noches["categoria_imc"] = pd.cut(noches["bmi"], bins=[0, 18.5, 25, 30, 100], right=False,
                                 labels=["Bajo peso", "Normal", "Sobrepeso", "Obesidad"])
for col in ["grupo_etario", "categoria_imc"]:
    print(f"{col}: {noches[col].value_counts(sort=False, dropna=False).to_dict()}")

Noches registradas por las personas con faltantes: {11: 1, 21: 1}
grupo_etario: {'18-30': 55, '31-45': 27, '46-60': 10, '>60': 35, nan: 1}
categoria_imc: {'Bajo peso': 1, 'Normal': 81, 'Sobrepeso': 38, 'Obesidad': 7, nan: 1}


### 2.3 Indicadores por noche

Para detectar noches atípicas se resume cada noche con indicadores de arquitectura del sueño (calculados sobre el consenso humano, que es el estándar de referencia), de calidad de señal y de concordancia:

| Indicador | Definición |
|---|---|
| `horas_registro` | Duración total del registro. |
| `horas_validas_psg` / `_hb` | Horas con etiqueta válida del consenso y de la IA de cada dispositivo. |
| `tst_min` | Tiempo total de sueño (N1 + N2 + N3 + REM). |
| `eficiencia` | `tst_min` / tiempo con etiqueta válida. El dataset no informa la hora de apagado de luces, así que se usa todo el registro. |
| `latencia_sueno_min` / `latencia_rem_min` | Desde el inicio del registro hasta la primera época de sueño, y desde ésta hasta la primera de REM. |
| `waso_min` | Vigilia entre la primera y la última época de sueño. |
| `pct_artefacto_hb` | % de épocas sin etiqueta de la vincha. |
| `kappa_psg` / `kappa_hb` | κ de Cohen entre cada IA y el consenso humano (épocas válidas en ambos). |

In [16]:
SUENO = [1, 2, 3, 4]   # N1, N2, N3, REM

def resumir_noche(g: pd.DataFrame) -> pd.Series:
    hum = g["stage_hum"]
    dormido = hum.isin(SUENO).to_numpy()
    r = {"horas_registro": len(g) / 120, "tst_min": dormido.sum() / 2,
         "eficiencia": dormido.sum() / hum.notna().sum(),
         "pct_artefacto_hb": g["stage_ai_hb"].isna().mean() * 100}
    if dormido.any():
        i0, i1 = dormido.argmax(), len(dormido) - 1 - dormido[::-1].argmax()
        r["latencia_sueno_min"] = i0 / 2 if i0 > 0 else np.nan   # Sin vigilia inicial → no calculable
        rem = (hum.to_numpy() == 4)
        r["latencia_rem_min"] = (rem[i0:].argmax()) / 2 if rem.any() else np.nan
        r["waso_min"] = (hum.iloc[i0:i1 + 1] == 0).sum() / 2
    for disp in ["psg", "hb"]:
        ia = g[f"stage_ai_{disp}"]
        ok = hum.notna() & ia.notna()
        r[f"horas_validas_{disp}"] = ok.sum() / 120
        r[f"kappa_{disp}"] = cohen_kappa_score(hum[ok].astype(int), ia[ok].astype(int))
    return pd.Series(r)

indicadores = epocas.groupby("participant_id", observed=True).apply(resumir_noche)
indicadores.describe().loc[["mean", "min", "50%", "max"]].round(2)

,horas_registro,tst_min,eficiencia,pct_artefacto_hb,latencia_sueno_min,latencia_rem_min,waso_min,horas_validas_psg,kappa_psg,horas_validas_hb,kappa_hb
mean,7.82,393.05,0.84,3.88,34.74,125.54,34.11,7.77,0.77,7.48,0.76
min,3.41,113.50,0.48,0.00,2.00,12.00,2.50,3.41,0.39,1.98,0.33
50%,7.79,402.00,0.86,0.23,27.00,102.25,19.00,7.72,0.81,7.50,0.79
max,9.32,512.00,0.97,73.99,175.00,362.00,198.50,9.32,0.93,9.10,0.92


### 2.4 Detección de noches atípicas

Aca se combinan dos enfoques vistos en la materia: **cercos de Tukey** (univariado, 1,5 × IQR) e **Isolation Forest** (multivariado, sobre los indicadores escalados con `RobustScaler`, que es resistente a los propios atípicos).

In [22]:
VARS_ATIPICOS = ["horas_registro", "tst_min", "eficiencia", "pct_artefacto_hb", "kappa_psg", "kappa_hb"]

def fuera_de_cercos(x: pd.Series, k: float = 1.5) -> pd.Series:
    q1, q3 = x.quantile([0.25, 0.75])
    return (x < q1 - k * (q3 - q1)) | (x > q3 + k * (q3 - q1))

tukey = indicadores[VARS_ATIPICOS].apply(fuera_de_cercos)
print("Noches fuera de los cercos de Tukey por indicador:")
for col in VARS_ATIPICOS:
    print(f"  {col:<17} {tukey[col].sum():>2} → {sorted(tukey.index[tukey[col]], key=lambda s: int(s[4:]))}")

X = RobustScaler().fit_transform(indicadores[VARS_ATIPICOS])
iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=SEED).fit(X)
atipicas_if = indicadores.index[iso.predict(X) == -1]
print(f"\nIsolation Forest (5 %): {sorted(atipicas_if, key=lambda s: int(s[4:]))}")
indicadores.loc[atipicas_if, VARS_ATIPICOS].round(2)

Noches fuera de los cercos de Tukey por indicador:
  horas_registro     2 → ['sub-8', 'sub-53']
  tst_min            6 → ['sub-8', 'sub-24', 'sub-34', 'sub-40', 'sub-53', 'sub-121']
  eficiencia         5 → ['sub-8', 'sub-24', 'sub-34', 'sub-79', 'sub-121']
  pct_artefacto_hb  17 → ['sub-1', 'sub-7', 'sub-11', 'sub-13', 'sub-20', 'sub-23', 'sub-29', 'sub-53', 'sub-55', 'sub-57', 'sub-61', 'sub-65', 'sub-72', 'sub-75', 'sub-76', 'sub-77', 'sub-88']
  kappa_psg          7 → ['sub-4', 'sub-10', 'sub-25', 'sub-96', 'sub-98', 'sub-121', 'sub-122']
  kappa_hb           6 → ['sub-4', 'sub-10', 'sub-96', 'sub-115', 'sub-121', 'sub-122']

Isolation Forest (5 %): ['sub-1', 'sub-4', 'sub-8', 'sub-10', 'sub-13', 'sub-53', 'sub-121']


,horas_registro,tst_min,eficiencia,pct_artefacto_hb,kappa_psg,kappa_hb
participant_id,,,,,,
sub-1,7.62,357.0,0.78,73.99,0.77,0.70
sub-10,8.28,471.5,0.95,0.00,0.39,0.33
sub-121,7.44,251.5,0.56,1.23,0.47,0.36
sub-13,8.31,465.5,0.93,54.56,0.60,0.61
sub-4,8.92,459.0,0.86,7.01,0.42,0.43
sub-53,5.55,228.5,0.70,8.41,0.76,0.57
sub-8,3.41,113.5,0.56,0.00,0.86,0.70


### 2.5 Decisiones tomadas sobre los atípicos

1. **Baja concordancia o mal dormir no son errores.** Las noches con κ (kappa) bajo o con poco tiempo de sueño (p. ej. `sub-4`, `sub-10`, `sub-121`) son justamente el fenómeno que se quiere estudiar. Eliminarlas sobreestimaría el desempeño de los algoritmos. **Decisión: *Se conservan*.**
2. **Criterio de señal mínima: al menos 4 h válidas.** Un κ calculado sobre pocas horas no es comparable con el de una noche completa, y los artefactos no ocurren al azar (se concentran en los movimientos y vigilias). Los cercos de Tukey no sirven para esta variable: `pct_artefacto_hb` es una variable muy asimétrica (mediana 0.2%), así que el cerco superior queda en ~8% y marca 17 noches con señal de sobra. Tukey supone una distribución aproximadamente simétrica. Por eso se usa un criterio absoluto: una noche es válida para un dispositivo solo si tiene ≥ 4 h de épocas válidas (`valida_psg` y `valida_hb`).
   - `sub-8` (registro de 3,4 h) queda excluida en ambos dispositivos.
   - `sub-1` (2,0 h válidas) y `sub-13` (3,8 h) quedan excluidas solo para la vincha.
   - `sub-53` (registro corto, 5,5 h) aparece como atípica en ambos métodos, pero supera el umbral y se conserva.
3. **Latencia no calculable.** `sub-1` comienza dormido, así que su latencia de sueño queda como `NA`.
4. **Ninguna noche se borra.** Las marcas viajan a la base transaccional y al almacén, donde el tablero las usa como filtro.

In [23]:
HORAS_MINIMAS = 4
indicadores["valida_psg"] = indicadores["horas_validas_psg"] >= HORAS_MINIMAS
indicadores["valida_hb"] = indicadores["horas_validas_hb"] >= HORAS_MINIMAS
print("Noches no válidas:",
      {d: sorted(indicadores.index[~indicadores[f"valida_{d}"]]) for d in ["psg", "hb"]})

noches = noches.merge(indicadores[["valida_psg", "valida_hb"]], left_on="participant_id",
                      right_index=True, how="left", validate="one_to_one")

epocas.to_parquet(DIR_PROCESSED / "02_epocas_limpias.parquet", index=False)
noches.to_parquet(DIR_PROCESSED / "02_noches.parquet", index=False)
print(f"Épocas: {epocas.shape} | Noches: {noches.shape} → guardadas en {DIR_PROCESSED}/")

Noches no válidas: {'psg': ['sub-8'], 'hb': ['sub-1', 'sub-13', 'sub-8']}
Épocas: (120095, 8) | Noches: (128, 9) → guardadas en data/processed/


---

## 3. Carga en la base de datos transaccional (SQLite)

La base transaccional (OLTP) es el sistema de registro del pipeline: Guarda cada dato una sola vez, garantiza la integridad referencial mediante llaves primarias y foráneas, y valida las reglas de negocio con restricciones (`CHECK`, `NOT NULL`). Las cargas se hacen dentro de transacciones que cumplen las propiedades **ACID**. A partir de esta base se construye después el almacén de datos.

### 3.1 Normalización

Se parte de la tabla única de la Transformación, cuya llave es (`participant_id`, `onset`):

| Forma normal | Problema en la tabla única | Acción |
|---|---|---|
| **1FN** | `stage_hum`, `stage_ai_psg` y `stage_ai_hb` son un **grupo repetitivo**: el mismo atributo (estadio) registrado por tres evaluadores distintos. | Se pasan a filas: una etiqueta por (sesión, época, evaluador). La llave compuesta pasa a ser {`sesion_id`, `nro_epoca`, `evaluador_id`}. |
| **2FN** | **Dependencias parciales**: edad, IMC, `pid` y las marcas de validez dependen solo de `sesion_id` (una parte de la llave); `onset_s`, de (`sesion_id`, `nro_epoca`); la descripción del evaluador, solo de `evaluador_id`. | Se extraen `SESION`, `EPOCA` y `EVALUADOR`. |
| **3FN** | **Dependencias transitivas**: `sesion_id → pid → sexo`; `evaluador_id → dispositivo → fabricante, modelo, fs`; `estadio_id → sigla, nombre`. | Se extraen `PERSONA`, `DISPOSITIVO` y `ESTADIO`. |

La configuración de sensores de cada noche (`channels.tsv`) también viene en formato ancho: una columna 0/1 por grupo de sensores, que es otro **grupo repetitivo**. Se lleva a filas y queda una relación **N:M** entre sesiones y sensores (una noche usa varios sensores y cada sensor se usa en muchas noches), que se resuelve con la tabla puente `SESION_SENSOR`. La descripción de cada sensor depende solo del sensor, así que va en su propia tabla `SENSOR` (3FN).

El grupo etario y la categoría de IMC no se guardan en la base transaccional: se deducen de la edad y del IMC (otra dependencia transitiva) y se calculan recién en el almacén.

### 3.2 Diagrama entidad-relación

![Diagrama entidad-relación de la base transaccional](https://github.com/aylinavch/ITBA-Introduccion-Ingenieria-de-Datos/blob/main/TP-Final/img/er_oltp.png?raw=true)

- Una **persona** puede tener varias **sesiones** (noches); cada **sesion** tiene muchas **epocas** de 30s de duración.
- Cada **epoca** recibe una **etiqueta** por **evaluador** (consenso humano, IA sobre PSG e IA sobre vincha). _Esta tabla resuelve la relación N:M entre épocas y evaluadores._
- Cada **sesion** usa varios **sensores** y cada **sensor** pertenece a un **dispositivo**. _La tabla puente `SESION_SENSOR` resuelve la relación N:M entre sesiones y sensores._
- Cada **evaluador** trabaja sobre la señal de un **dispositivo**. La **etiqueta** referencia a un **estadio**, o queda en `NULL` si la época no tiene etiqueta válida.

In [24]:
rutas_json = [f"{s}/eeg/{s}_task-Sleep_acq-{a}_eeg.json" for s in SESIONES for a in ADQUISICIONES]
with ThreadPoolExecutor(max_workers=8) as pool:
    metadatos = [pd.read_json(ruta, typ="series") for ruta in pool.map(descargar, rutas_json)]

meta = pd.DataFrame(metadatos).assign(acq=[a for _ in SESIONES for a in ADQUISICIONES])
# EEGChannelCount se excluye: en el PSG varía entre noches (8 a 13 canales), depende de la sesión
canales = meta.groupby("acq")["EEGChannelCount"].unique()
print("Cantidad de canales observada por dispositivo:", {k: sorted(map(int, v)) for k, v in canales.items()})
COLS_DISP = ["Manufacturer", "ManufacturersModelName", "SamplingFrequency"]
assert (meta.groupby("acq")[COLS_DISP].nunique() == 1).all().all(), "Los metadatos varían entre sesiones"

dispositivo = (meta.drop_duplicates("acq")[["acq"] + COLS_DISP]
               .rename(columns={"Manufacturer": "fabricante", "ManufacturersModelName": "modelo",
                                "SamplingFrequency": "fs_hz"})
               .reset_index(drop=True))
dispositivo.insert(0, "dispositivo_id", dispositivo.index + 1)
dispositivo

Cantidad de canales observada por dispositivo: {'headband': [2], 'psg': [8, 9, 11, 12, 13]}


,dispositivo_id,acq,fabricante,modelo,fs_hz
0,1,psg,Micromed,SD plus,256
1,2,headband,Bitbrain Technologies,IKON Sleep,256


Los atributos de cada dispositivo se leen de los metadatos BIDS (`*_eeg.json`). Antes de usarlos se verifica que sean idénticos en las 128 noches, es decir, que dependen del dispositivo y no de la sesión. La verificación excluye `EEGChannelCount` ya que en el PSG la cantidad de canales cambia según los sensores colocados cada noche (de 8 a 13), así que depende de la sesión y no se guarda en `DISPOSITIVO`. Esa variación queda representada en la tabla puente `SESION_SENSOR`.

**Sensores por sesión (`channels.tsv`).** El archivo tiene una fila por noche y una columna 0/1 por grupo de sensores. Su diccionario (`channels.json`) describe cada grupo. El prefijo del código (`PSG_` o `HB_`) indica a qué dispositivo pertenece el sensor.

In [ ]:
descripcion_sensores = pd.read_json(descargar("channels.json"), typ="series")
COLS_SENSOR = [c for c in descripcion_sensores.index if c != "REC"]
canales = pd.read_csv(descargar("channels.tsv"), sep="\t", na_values=["n/a"],
                      dtype={"REC": "string", **{c: "Int8" for c in COLS_SENSOR}})

assert set(canales["REC"]) == set(SESIONES), "channels.tsv no coincide con las sesiones"
assert canales[COLS_SENSOR].isin([0, 1]).all().all(), "Valores fuera del dominio {0, 1}"

ACQ_POR_PREFIJO = {"PSG": "psg", "HB": "headband"}
sensor = pd.DataFrame({
    "sensor_id": range(1, len(COLS_SENSOR) + 1),
    "codigo": COLS_SENSOR,
    "dispositivo_id": [dispositivo.set_index("acq").loc[ACQ_POR_PREFIJO[c.split("_")[0]], "dispositivo_id"]
                       for c in COLS_SENSOR],
    "descripcion": [descripcion_sensores[c]["Description"] for c in COLS_SENSOR],
})

# Formato ancho → largo: una fila por (sesión, sensor presente)
sesion_sensor = (canales.melt(id_vars="REC", var_name="codigo", value_name="presente")
                 .query("presente == 1")
                 .merge(sensor[["codigo", "sensor_id"]], on="codigo")
                 .rename(columns={"REC": "sesion_id"})[["sesion_id", "sensor_id"]])

print(f"{len(sensor)} sensores | {len(sesion_sensor)} filas en la tabla puente | "
      f"{canales[COLS_SENSOR].drop_duplicates().shape[0]} configuraciones distintas de sensores")
print("Noches con cada sensor:", {c: int(n) for c, n in canales[COLS_SENSOR].sum().items()})

### 3.3 Esquema (DDL)

Cada tabla declara su llave primaria, sus llaves foráneas y las restricciones de dominio. `PRAGMA foreign_keys = ON` es necesario porque SQLite no aplica la integridad referencial por defecto. La base se recrea en cada ejecución para que el cuaderno sea reproducible.

In [25]:
DIR_DB = Path("data/db")
DIR_DB.mkdir(parents=True, exist_ok=True)
RUTA_OLTP = DIR_DB / "boas_oltp.db"
RUTA_OLTP.unlink(missing_ok=True)

DDL_OLTP = """
CREATE TABLE persona (
    pid         INTEGER PRIMARY KEY,
    sexo        TEXT    NOT NULL CHECK (sexo IN ('F', 'M'))
);
CREATE TABLE sesion (
    sesion_id   TEXT    PRIMARY KEY,
    pid         INTEGER NOT NULL REFERENCES persona(pid),
    edad        INTEGER CHECK (edad BETWEEN 18 AND 120),          -- NULL: dato no informado
    imc         REAL    CHECK (imc BETWEEN 10 AND 60),            -- NULL: dato no informado
    valida_psg  INTEGER NOT NULL CHECK (valida_psg IN (0, 1)),    -- >= 4 h de señal válida
    valida_hb   INTEGER NOT NULL CHECK (valida_hb IN (0, 1))
);
CREATE TABLE dispositivo (
    dispositivo_id INTEGER PRIMARY KEY,
    acq         TEXT    NOT NULL UNIQUE,
    fabricante  TEXT    NOT NULL,
    modelo      TEXT    NOT NULL,
    fs_hz       INTEGER NOT NULL CHECK (fs_hz > 0)
);
CREATE TABLE sensor (
    sensor_id      INTEGER PRIMARY KEY,
    codigo         TEXT    NOT NULL UNIQUE,
    dispositivo_id INTEGER NOT NULL REFERENCES dispositivo(dispositivo_id),
    descripcion    TEXT    NOT NULL
);
CREATE TABLE sesion_sensor (                                      -- tabla puente (N:M)
    sesion_id   TEXT    NOT NULL REFERENCES sesion(sesion_id),
    sensor_id   INTEGER NOT NULL REFERENCES sensor(sensor_id),
    PRIMARY KEY (sesion_id, sensor_id)
);
CREATE TABLE evaluador (
    evaluador_id   INTEGER PRIMARY KEY,
    descripcion    TEXT    NOT NULL UNIQUE,
    tipo           TEXT    NOT NULL CHECK (tipo IN ('humano', 'IA')),
    dispositivo_id INTEGER NOT NULL REFERENCES dispositivo(dispositivo_id)
);
CREATE TABLE estadio (
    estadio_id  INTEGER PRIMARY KEY,                              -- código original de BOAS
    sigla       TEXT    NOT NULL UNIQUE,
    nombre      TEXT    NOT NULL,
    es_sueno    INTEGER NOT NULL CHECK (es_sueno IN (0, 1))
);
CREATE TABLE epoca (
    sesion_id   TEXT    NOT NULL REFERENCES sesion(sesion_id),
    nro_epoca   INTEGER NOT NULL CHECK (nro_epoca >= 0),
    onset_s     INTEGER NOT NULL,
    PRIMARY KEY (sesion_id, nro_epoca)
);
CREATE TABLE etiqueta (
    sesion_id    TEXT    NOT NULL,
    nro_epoca    INTEGER NOT NULL,
    evaluador_id INTEGER NOT NULL REFERENCES evaluador(evaluador_id),
    estadio_id   INTEGER REFERENCES estadio(estadio_id),          -- NULL: desconexión o artefacto
    PRIMARY KEY (sesion_id, nro_epoca, evaluador_id),
    FOREIGN KEY (sesion_id, nro_epoca) REFERENCES epoca(sesion_id, nro_epoca)
);
"""

conn_oltp = sqlite3.connect(RUTA_OLTP)
conn_oltp.execute("PRAGMA foreign_keys = ON")
conn_oltp.executescript(DDL_OLTP)
print("Tablas creadas:", [t for (t,) in conn_oltp.execute(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY rowid")])

Tablas creadas: ['persona', 'sesion', 'dispositivo', 'evaluador', 'estadio', 'epoca', 'etiqueta']


### 3.4 Carga

Se arma un DataFrame por tabla y se insertan todos en una sola transacción (`with conn:`). Si cualquier fila viola una restricción, se revierte la carga completa y la base nunca queda a medio cargar (atomicidad y consistencia). El orden de inserción respeta las llaves foráneas: primero las tablas referenciadas.

In [26]:
# PERSONA: el sexo debe ser único por persona (verificación de la dependencia pid → sexo)
persona = noches[["pid", "sex"]].drop_duplicates().rename(columns={"sex": "sexo"})
assert persona["pid"].is_unique

sesion = (noches.rename(columns={"participant_id": "sesion_id", "age": "edad", "bmi": "imc"})
          [["sesion_id", "pid", "edad", "imc", "valida_psg", "valida_hb"]]
          .assign(imc=lambda d: d["imc"].astype("float64").round(1)))

evaluador = pd.DataFrame({
    "evaluador_id": [1, 2, 3],
    "descripcion": ["Consenso de 3 expertos (AASM)", "IA sobre PSG", "IA sobre vincha"],
    "tipo": ["humano", "IA", "IA"],
    "dispositivo_id": dispositivo.set_index("acq").loc[["psg", "psg", "headband"], "dispositivo_id"].to_list(),
})
estadio = pd.DataFrame({
    "estadio_id": [0, 1, 2, 3, 4],
    "sigla": ["W", "N1", "N2", "N3", "REM"],
    "nombre": ["Vigilia", "Sueño NREM 1", "Sueño NREM 2", "Sueño NREM 3 (profundo)", "Sueño REM"],
    "es_sueno": [0, 1, 1, 1, 1],
})
epoca = epocas[["participant_id", "epoca", "onset"]].rename(
    columns={"participant_id": "sesion_id", "epoca": "nro_epoca", "onset": "onset_s"})

# ETIQUETA: el grupo repetitivo de 3 columnas pasa a filas (1FN)
EVALUADOR_POR_COLUMNA = {"stage_hum": 1, "stage_ai_psg": 2, "stage_ai_hb": 3}
etiqueta = (epocas.melt(id_vars=["participant_id", "epoca"], value_vars=COLS_ESTADIO,
                        var_name="columna", value_name="estadio_id")
            .assign(evaluador_id=lambda d: d["columna"].map(EVALUADOR_POR_COLUMNA))
            .rename(columns={"participant_id": "sesion_id", "epoca": "nro_epoca"})
            [["sesion_id", "nro_epoca", "evaluador_id", "estadio_id"]])

def insertar(conn: sqlite3.Connection, tabla: str, df: pd.DataFrame) -> None:
    """Inserta un DataFrame convirtiendo los faltantes (NA/NaN) en NULL."""
    columnas = [[None if pd.isna(v) else v for v in df[c].tolist()] for c in df.columns]
    sql = f"INSERT INTO {tabla} ({', '.join(df.columns)}) VALUES ({', '.join('?' * df.shape[1])})"
    conn.executemany(sql, zip(*columnas))

TABLAS_OLTP = {"persona": persona, "sesion": sesion, "dispositivo": dispositivo,
               "sensor": sensor, "sesion_sensor": sesion_sensor, "evaluador": evaluador, "estadio": estadio, "epoca": epoca, "etiqueta": etiqueta}
with conn_oltp:   # Transacción: confirma todo (COMMIT) o revierte todo (ROLLBACK)
    for nombre, df in TABLAS_OLTP.items():
        insertar(conn_oltp, nombre, df)

for nombre in TABLAS_OLTP:
    n = conn_oltp.execute(f"SELECT COUNT(*) FROM {nombre}").fetchone()[0]
    print(f"  {nombre:<14} {n:>8,} filas")
print(f"Tamaño de la base: {RUTA_OLTP.stat().st_size / 1024**2:.1f} MB")

  persona           100 filas
  sesion            128 filas
  dispositivo         2 filas
  evaluador           3 filas
  estadio             5 filas
  epoca         120,095 filas
  etiqueta      360,285 filas
Tamaño de la base: 19.1 MB


### 3.5 Validación

1. **Integridad referencial:** `PRAGMA foreign_key_check` no debe devolver filas huérfanas.
2. **Consistencia:** La base debe rechazar filas que violan las restricciones.
3. **Ida y vuelta:** Al reconstruir la tabla de épocas desde la base con `JOIN` y una pivot en SQL, el resultado debe ser idéntico a lo que se cargó. Lo mismo se verifica para la configuración de sensores: la tabla puente, vuelta a formato ancho, debe reproducir `channels.tsv`. Esto demuestra que la normalización no perdió información.

In [27]:
assert conn_oltp.execute("PRAGMA foreign_key_check").fetchall() == []
print("✓ Sin violaciones de integridad referencial")

pruebas = {
    "estadio inexistente (7)":      "UPDATE etiqueta SET estadio_id = 7 "
                                    "WHERE sesion_id = 'sub-1' AND nro_epoca = 0 AND evaluador_id = 1",
    "época de una sesión inexistente": "INSERT INTO epoca VALUES ('sub-999', 0, 0)",
    "sexo fuera del dominio":       "INSERT INTO persona VALUES (999, 'X')",
}
for caso, sql in pruebas.items():
    try:
        with conn_oltp:
            conn_oltp.execute(sql)
        print(f"✗ Se aceptó: {caso}")
    except sqlite3.IntegrityError as error:
        print(f"✓ Rechazado ({caso}): {error}")

SQL_RECONSTRUCCION = """
SELECT e.sesion_id AS participant_id, e.onset_s AS onset,
       MAX(CASE WHEN t.evaluador_id = 1 THEN t.estadio_id END) AS stage_hum,
       MAX(CASE WHEN t.evaluador_id = 2 THEN t.estadio_id END) AS stage_ai_psg,
       MAX(CASE WHEN t.evaluador_id = 3 THEN t.estadio_id END) AS stage_ai_hb
FROM epoca AS e
JOIN etiqueta AS t USING (sesion_id, nro_epoca)
GROUP BY e.sesion_id, e.nro_epoca
"""
reconstruida = pd.read_sql(SQL_RECONSTRUCCION, conn_oltp)
original = epocas[["participant_id", "onset"] + COLS_ESTADIO].astype({"participant_id": str})
clave = ["participant_id", "onset"]
pd.testing.assert_frame_equal(
    reconstruida.sort_values(clave).reset_index(drop=True).astype({c: "Int8" for c in COLS_ESTADIO}),
    original.sort_values(clave).reset_index(drop=True).astype({c: "Int8" for c in COLS_ESTADIO}),
    check_dtype=False)
print(f"✓ Ida y vuelta: las {len(reconstruida):,} épocas reconstruidas desde SQL coinciden con las cargadas")

sensores_sql = pd.read_sql("""
    SELECT ss.sesion_id AS REC, s.codigo
    FROM sesion_sensor AS ss JOIN sensor AS s USING (sensor_id)""", conn_oltp)
ancho = (pd.crosstab(sensores_sql["REC"], sensores_sql["codigo"])
         .reindex(index=canales["REC"], columns=COLS_SENSOR, fill_value=0))
assert (ancho.to_numpy() == canales[COLS_SENSOR].to_numpy()).all()
print("✓ Ida y vuelta: la tabla puente reproduce channels.tsv")

✓ Sin violaciones de integridad referencial
✓ Rechazado (estadio inexistente (7)): FOREIGN KEY constraint failed
✓ Rechazado (época de una sesión inexistente): FOREIGN KEY constraint failed
✓ Rechazado (sexo fuera del dominio): CHECK constraint failed: sexo IN ('F', 'M')
✓ Ida y vuelta: las 120,095 épocas reconstruidas desde SQL coinciden con las cargadas


---